# Pipeline Performance Engineering

<a href="https://colab.research.google.com/github/rubenfonnegra/HPC/blob/master/Sem_06/Performance_Engineering_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/HPC/tree/master/Sem_06/Performance_Engineering_Lab.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


Instead of benchmarking isolated mathematical operations, we ask:

> **Where is time and memory actually being spent while loading, cleaning, transforming, aggregating, and exporting data?**

The workflow is:

$$
\boxed{
\text{Baseline}
\rightarrow
\text{Measure}
\rightarrow
\text{Decompose}
\rightarrow
\text{Profile}
\rightarrow
\text{Diagnose}
\rightarrow
\text{Hypothesize}
\rightarrow
\text{Optimize}
\rightarrow
\text{Measure Again}
}
$$

The final artifact is a **Performance Profile v1**.


**Important:** benchmark results depend on the machine, operating system, CPU, GPU, BLAS implementation, system load, and memory hierarchy. Exact numbers will vary. The goal is to interpret the behavior, not reproduce a specific number.

---

> **📘 High Performance Computing (HPC)**
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution



## Case study

A manufacturing company receives telemetry from hundreds of industrial devices.

Each record contains:

- timestamp;
- device and site;
- device type;
- temperature;
- vibration;
- pressure;
- power consumption;
- operating status;
- maintenance and failure labels.

The raw CSV deliberately contains realistic data-quality issues:

- missing values;
- inconsistent capitalization and whitespace;
- duplicated records;
- rare numerical outliers;
- timestamps stored as text.

Our analytical goal is to build a daily KPI table by **site** and **device type** and identify high-risk observations.

The performance question is:

> **Can we make this pipeline faster and more memory-efficient without changing its analytical result?**

## Setup and environment

In [ ]:
import os
import sys
import time
import json
import platform
import tracemalloc
import cProfile
import pstats
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("industrial_iot.csv")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CPU cores:", os.cpu_count())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Dataset exists:", DATA_PATH.exists())

Download [`industrial_iot.csv`](https://drive.google.com/file/d/1gVnVLTj8SiiVUAbunhDVdruPNw__sQVv/view?usp=sharing) before continuing.

In [ ]:
if DATA_PATH.exists():
    size_mb = DATA_PATH.stat().st_size / 1024**2
    print(f"CSV size: {size_mb:.2f} MB")
else:
    print("Dataset not found. Upload the CSV and update DATA_PATH.")

## 1. First look: data profiling before performance profiling

Before optimizing anything, understand the data.

Questions:

1. How many rows and columns are there?
2. Which columns have missing values?
3. Which columns have inconsistent categories?
4. Are there duplicates?
5. What is the raw memory footprint after loading?

In [ ]:
sample = pd.read_csv(DATA_PATH, nrows=10)
sample

In [ ]:
sample.dtypes

## 2. Baseline hypothesis

We will first create a **reasonable** pipeline.

It will:

1. load the entire CSV with default type inference;
2. convert the timestamp after loading;
3. normalize text columns;
4. remove duplicates;
5. fill missing numeric values with medians;
6. create a row-wise risk label;
7. aggregate daily KPIs;
8. export the result.

This becomes our **baseline**.

We will measure:

$$
T_{\text{total}}
=
T_{\text{load}}
+
T_{\text{clean}}
+
T_{\text{features}}
+
T_{\text{aggregate}}
+
T_{\text{export}}
$$

and track:

- wall-clock time;
- rows/s;
- DataFrame memory footprint;
- peak Python memory allocation;
- output equivalence.

In [ ]:
def dataframe_memory_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

def run_with_peak_memory(func, *args, **kwargs):
    tracemalloc.start()
    start = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - start
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, elapsed, peak / 1024**2

## 3. Baseline pipeline

In [ ]:
def baseline_pipeline(path, export_path="baseline_daily_kpis.csv"):
    stage = {}
    total_start = time.perf_counter()

    # 1. LOAD
    t0 = time.perf_counter()
    df = pd.read_csv(path)
    stage["load"] = time.perf_counter() - t0

    raw_memory_mb = dataframe_memory_mb(df)

    # 2. CLEAN
    t0 = time.perf_counter()

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Python-level apply: intentionally reasonable but not optimal
    df["site"] = df["site"].apply(
        lambda x: x.strip().lower() if isinstance(x, str) else x
    )
    df["status"] = df["status"].apply(
        lambda x: x.strip().lower() if isinstance(x, str) else x
    )
    df["device_type"] = df["device_type"].apply(
        lambda x: x.strip().lower() if isinstance(x, str) else x
    )

    df = df.drop_duplicates(
        subset=[
            "timestamp", "device_id", "site", "device_type",
            "temperature_c", "vibration_mm_s", "pressure_bar",
            "power_kw", "status"
        ]
    )

    numeric_cols = [
        "temperature_c", "vibration_mm_s",
        "pressure_bar", "power_kw"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

    stage["clean"] = time.perf_counter() - t0

    # 3. FEATURE ENGINEERING
    t0 = time.perf_counter()

    def risk_label(row):
        score = 0
        if row["temperature_c"] > 45:
            score += 1
        if row["vibration_mm_s"] > 7:
            score += 1
        if row["power_kw"] > 35:
            score += 1
        if row["status"] == "critical":
            score += 2

        if score >= 3:
            return "high"
        elif score >= 1:
            return "medium"
        return "low"

    df["risk_level"] = df.apply(risk_label, axis=1)
    df["date"] = df["timestamp"].dt.date

    stage["features"] = time.perf_counter() - t0

    # 4. KPIs
    t0 = time.perf_counter()

    daily = (
        df.groupby(
            ["date", "site", "device_type"],
            dropna=False
        )
        .agg(
            records=("record_id", "size"),
            mean_temperature_c=("temperature_c", "mean"),
            p95_vibration_mm_s=(
                "vibration_mm_s",
                lambda x: x.quantile(0.95)
            ),
            mean_power_kw=("power_kw", "mean"),
            failure_rate=("failure_within_24h", "mean"),
            maintenance_rate=("maintenance_flag", "mean"),
            high_risk_records=(
                "risk_level",
                lambda x: (x == "high").sum()
            )
        )
        .reset_index()
    )

    stage["aggregate"] = time.perf_counter() - t0

    # 5. EXPORT
    t0 = time.perf_counter()
    daily.to_csv(export_path, index=False)
    stage["export"] = time.perf_counter() - t0

    stage["total"] = time.perf_counter() - total_start

    return {
        "data": df,
        "daily": daily,
        "stage_times": stage,
        "dataframe_memory_mb": dataframe_memory_mb(df),
        "raw_memory_mb": raw_memory_mb,
        "rows": len(df),
    }

### Prediction before running

Write down your hypothesis:

- Which stage will be slowest?
- Which operation is likely to allocate the most memory?

In [ ]:
baseline_result, baseline_elapsed, baseline_peak_mb = run_with_peak_memory(
    baseline_pipeline,
    DATA_PATH
)

baseline_result["stage_times"], baseline_peak_mb

In [ ]:
baseline_summary = pd.DataFrame({
    "stage": list(baseline_result["stage_times"].keys()),
    "seconds": list(baseline_result["stage_times"].values())
})

baseline_summary

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(
    baseline_summary["stage"],
    baseline_summary["seconds"]
)
plt.ylabel("Seconds")
plt.title("Baseline runtime decomposition")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
baseline_metrics = {
    "wall_clock_s": baseline_result["stage_times"]["total"],
    "rows_processed": baseline_result["rows"],
    "throughput_rows_s": (
        baseline_result["rows"] /
        baseline_result["stage_times"]["total"]
    ),
    "dataframe_memory_mb": baseline_result["dataframe_memory_mb"],
    "peak_python_alloc_mb": baseline_peak_mb,
}

pd.Series(baseline_metrics)

## 4. Profile the baseline

Timing tells us **how long**.

Profiling helps answer:

> **Where is Python spending that time?**

In [ ]:
profiler = cProfile.Profile()
profiler.enable()

_ = baseline_pipeline(
    DATA_PATH,
    export_path="baseline_profile_run.csv"
)

profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.sort_stats("cumulative")
stats.print_stats(20)

print(stream.getvalue())

## Diagnosis questions

1. Which stage dominates wall-clock time?
2. Which functions dominate cumulative profiler time?
3. Does the evidence point to:
   - I/O?
   - Python row-wise processing?
   - aggregation?
   - memory?
4. What optimization should be tested first?

Do not optimize everything at once.

Form one hypothesis.

# 5. Optimization hypotheses

We will test several evidence-based hypotheses.

### H1 — Reduce I/O and memory pressure

If we only load columns needed for the KPI computation and provide explicit dtypes, then:

- CSV parsing may become faster;
- memory footprint should decrease;
- later transformations should touch less data.

### H2 — Replace row-wise Python with vectorized operations

If risk classification uses NumPy/Pandas vectorization rather than `.apply(axis=1)`, feature engineering should become substantially faster.

### H3 — Use categorical dtypes

Low-cardinality text columns such as `site`, `device_type`, and `status` repeat many values. Converting them to `category` should reduce memory and may improve groupby performance.

### H4 — Vectorize expensive groupby logic

Instead of testing `"high"` inside a groupby lambda, create a numerical indicator once:

$$
\text{is\_high\_risk}
=
\begin{cases}
1 & \text{if high risk}\\
0 & \text{otherwise}
\end{cases}
$$

and aggregate it with `sum`.

# 6. Optimized pipeline

In [ ]:
def optimized_pipeline(path, export_path="optimized_daily_kpis.csv"):
    stage = {}
    total_start = time.perf_counter()

    usecols = [
        "record_id", "timestamp", "device_id", "site", "device_type",
        "temperature_c", "vibration_mm_s", "pressure_bar", "power_kw",
        "status", "maintenance_flag", "failure_within_24h"
    ]

    dtype_map = {
        "record_id": "int32",
        "device_id": "string",
        "site": "string",
        "device_type": "string",
        "temperature_c": "float32",
        "vibration_mm_s": "float32",
        "pressure_bar": "float32",
        "power_kw": "float32",
        "status": "string",
        "maintenance_flag": "int8",
        "failure_within_24h": "int8",
    }

    # 1. LOAD
    t0 = time.perf_counter()

    df = pd.read_csv(
        path,
        usecols=usecols,
        dtype=dtype_map,
        parse_dates=["timestamp"]
    )

    stage["load"] = time.perf_counter() - t0
    raw_memory_mb = dataframe_memory_mb(df)

    # 2. CLEAN
    t0 = time.perf_counter()

    # Vectorized string normalization
    for col in ["site", "device_type", "status"]:
        df[col] = df[col].str.strip().str.lower()

    df = df.drop_duplicates(
        subset=[
            "timestamp", "device_id", "site", "device_type",
            "temperature_c", "vibration_mm_s", "pressure_bar",
            "power_kw", "status"
        ]
    )

    numeric_cols = [
        "temperature_c", "vibration_mm_s",
        "pressure_bar", "power_kw"
    ]

    medians = df[numeric_cols].median()
    df[numeric_cols] = df[numeric_cols].fillna(medians)

    # Low-cardinality strings → category
    for col in ["site", "device_type", "status"]:
        df[col] = df[col].astype("category")

    stage["clean"] = time.perf_counter() - t0

    # 3. FEATURE ENGINEERING
    t0 = time.perf_counter()

    score = (
        (df["temperature_c"] > 45).astype("int8")
        + (df["vibration_mm_s"] > 7).astype("int8")
        + (df["power_kw"] > 35).astype("int8")
        + 2 * (df["status"] == "critical").astype("int8")
    )

    conditions = [
        score >= 3,
        score >= 1
    ]

    df["risk_level"] = pd.Categorical(
        np.select(
            conditions,
            ["high", "medium"],
            default="low"
        ),
        categories=["low", "medium", "high"]
    )

    df["is_high_risk"] = (
        df["risk_level"] == "high"
    ).astype("int8")

    # datetime64 day instead of Python date objects
    df["date"] = df["timestamp"].dt.floor("D")

    stage["features"] = time.perf_counter() - t0

    # 4. AGGREGATE
    t0 = time.perf_counter()

    # Pre-compute p95 separately; keep other aggregations native
    base_agg = (
        df.groupby(
            ["date", "site", "device_type"],
            observed=True,
            sort=False
        )
        .agg(
            records=("record_id", "size"),
            mean_temperature_c=("temperature_c", "mean"),
            mean_power_kw=("power_kw", "mean"),
            failure_rate=("failure_within_24h", "mean"),
            maintenance_rate=("maintenance_flag", "mean"),
            high_risk_records=("is_high_risk", "sum"),
        )
    )

    p95 = (
        df.groupby(
            ["date", "site", "device_type"],
            observed=True,
            sort=False
        )["vibration_mm_s"]
        .quantile(0.95)
        .rename("p95_vibration_mm_s")
    )

    daily = (
        base_agg
        .join(p95)
        .reset_index()
    )

    stage["aggregate"] = time.perf_counter() - t0

    # 5. EXPORT
    t0 = time.perf_counter()
    daily.to_csv(export_path, index=False)
    stage["export"] = time.perf_counter() - t0

    stage["total"] = time.perf_counter() - total_start

    return {
        "data": df,
        "daily": daily,
        "stage_times": stage,
        "dataframe_memory_mb": dataframe_memory_mb(df),
        "raw_memory_mb": raw_memory_mb,
        "rows": len(df),
    }

In [ ]:
optimized_result, optimized_elapsed, optimized_peak_mb = run_with_peak_memory(
    optimized_pipeline,
    DATA_PATH
)

optimized_result["stage_times"], optimized_peak_mb

## 7. Comparison - Before vs after

In [ ]:
comparison = pd.DataFrame({
    "metric": [
        "total_time_s",
        "throughput_rows_s",
        "dataframe_memory_mb",
        "peak_python_alloc_mb"
    ],
    "baseline": [
        baseline_result["stage_times"]["total"],
        baseline_result["rows"] / baseline_result["stage_times"]["total"],
        baseline_result["dataframe_memory_mb"],
        baseline_peak_mb,
    ],
    "optimized": [
        optimized_result["stage_times"]["total"],
        optimized_result["rows"] / optimized_result["stage_times"]["total"],
        optimized_result["dataframe_memory_mb"],
        optimized_peak_mb,
    ]
})

comparison["improvement_factor"] = [
    comparison.loc[0, "baseline"] / comparison.loc[0, "optimized"],
    comparison.loc[1, "optimized"] / comparison.loc[1, "baseline"],
    comparison.loc[2, "baseline"] / comparison.loc[2, "optimized"],
    comparison.loc[3, "baseline"] / comparison.loc[3, "optimized"],
]

comparison

In [ ]:
stage_compare = pd.DataFrame({
    "stage": [
        "load", "clean", "features",
        "aggregate", "export", "total"
    ],
    "baseline": [
        baseline_result["stage_times"][s]
        for s in [
            "load", "clean", "features",
            "aggregate", "export", "total"
        ]
    ],
    "optimized": [
        optimized_result["stage_times"][s]
        for s in [
            "load", "clean", "features",
            "aggregate", "export", "total"
        ]
    ],
})

stage_compare

In [ ]:
x = np.arange(len(stage_compare))
width = 0.38

plt.figure(figsize=(10, 5))
plt.bar(
    x - width/2,
    stage_compare["baseline"],
    width,
    label="Baseline"
)
plt.bar(
    x + width/2,
    stage_compare["optimized"],
    width,
    label="Optimized"
)
plt.xticks(x, stage_compare["stage"], rotation=20)
plt.ylabel("Seconds")
plt.title("Pipeline stage benchmarking")
plt.legend()
plt.grid(axis="y", alpha=0.2)
plt.show()

## 8. Validate analytical equivalence

Performance engineering is not successful if the faster pipeline changes the answer.

We therefore compare the baseline and optimized KPI outputs.

In [ ]:
base_daily = baseline_result["daily"].copy()
opt_daily = optimized_result["daily"].copy()

# Normalize date representation for comparison
base_daily["date"] = pd.to_datetime(base_daily["date"])
opt_daily["date"] = pd.to_datetime(opt_daily["date"])

keys = ["date", "site", "device_type"]

merged = base_daily.merge(
    opt_daily,
    on=keys,
    how="outer",
    suffixes=("_base", "_opt"),
    indicator=True
)

print("Rows baseline:", len(base_daily))
print("Rows optimized:", len(opt_daily))
print("Merge status:")
print(merged["_merge"].value_counts())

In [ ]:
numeric_metrics = [
    "records",
    "mean_temperature_c",
    "p95_vibration_mm_s",
    "mean_power_kw",
    "failure_rate",
    "maintenance_rate",
    "high_risk_records"
]

validation = []

for metric in numeric_metrics:
    a = merged[f"{metric}_base"].astype(float)
    b = merged[f"{metric}_opt"].astype(float)

    max_abs_diff = np.nanmax(np.abs(a - b))

    validation.append({
        "metric": metric,
        "max_absolute_difference": max_abs_diff
    })

pd.DataFrame(validation)

Is it really valid? Depends 🤷🏽‍♂️

## 10. Memory profiling by column

Which columns consume the most memory?

In [ ]:
baseline_memory_by_col = (
    baseline_result["data"]
    .memory_usage(deep=True)
    .sort_values(ascending=False)
    / 1024**2
)

optimized_memory_by_col = (
    optimized_result["data"]
    .memory_usage(deep=True)
    .sort_values(ascending=False)
    / 1024**2
)

memory_compare = pd.concat(
    [
        baseline_memory_by_col.rename("baseline_mb"),
        optimized_memory_by_col.rename("optimized_mb")
    ],
    axis=1
).fillna(0)

memory_compare["saved_mb"] = (
    memory_compare["baseline_mb"]
    - memory_compare["optimized_mb"]
)

memory_compare.sort_values(
    "saved_mb",
    ascending=False
).head(15)

## 11. Load only what you need

Suppose a downstream analysis needs only:

- timestamp;
- site;
- device type;
- temperature;
- vibration;
- power;
- status.

Benchmark loading **all columns** versus only those columns.

This tests an I/O hypothesis directly.

In [ ]:
minimal_cols = [
    "timestamp", "site", "device_type",
    "temperature_c", "vibration_mm_s",
    "power_kw", "status"
]

def load_all():
    return pd.read_csv(DATA_PATH)

def load_minimal():
    return pd.read_csv(
        DATA_PATH,
        usecols=minimal_cols
    )

_, all_load_time, all_peak = run_with_peak_memory(load_all)
_, min_load_time, min_peak = run_with_peak_memory(load_minimal)

pd.DataFrame([
    {
        "strategy": "all columns",
        "load_time_s": all_load_time,
        "peak_python_alloc_mb": all_peak
    },
    {
        "strategy": "minimal columns",
        "load_time_s": min_load_time,
        "peak_python_alloc_mb": min_peak
    }
])

## 13. Performance Profile v1 for data pipeline

Your final diagnosis should be evidence-based.

Complete the structure below for your own research pipeline.

In [ ]:
performance_profile_v1 = {
    "workload": {
        "dataset": str(DATA_PATH),
        "dataset_size_mb": (
            round(DATA_PATH.stat().st_size / 1024**2, 2)
            if DATA_PATH.exists() else None
        ),
        "rows": baseline_result["rows"],
        "analytical_goal": "Daily industrial telemetry KPI generation"
    },

    "baseline": {
        "total_time_s": baseline_result["stage_times"]["total"],
        "throughput_rows_s": (
            baseline_result["rows"] /
            baseline_result["stage_times"]["total"]
        ),
        "dataframe_memory_mb": baseline_result["dataframe_memory_mb"],
        "peak_python_alloc_mb": baseline_peak_mb,
        "stage_times": baseline_result["stage_times"]
    },

    "optimized": {
        "total_time_s": optimized_result["stage_times"]["total"],
        "throughput_rows_s": (
            optimized_result["rows"] /
            optimized_result["stage_times"]["total"]
        ),
        "dataframe_memory_mb": optimized_result["dataframe_memory_mb"],
        "peak_python_alloc_mb": optimized_peak_mb,
        "stage_times": optimized_result["stage_times"]
    }
}

performance_profile_v1

In [ ]:
with open(
    "performance_profile_v1_data_pipeline.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        performance_profile_v1,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Exported performance_profile_v1_data_pipeline.json")

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Computación Aplicada (HPC) courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Pipeline Performance Engineering: HPC Notebook*. Institución Universitaria Pascual Bravo.